# B2-023-generative-models-diffusion — Practice p02 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** generative-adversarial-network

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

**Answer: A.**

*Why A is right.* The discriminator maximizes $V$, and with batch means $-\hat V=\mathrm{BCE}(\ell_{\text{real}},1)+\mathrm{BCE}(\ell_{\text{fake}},0)$, the **sum** of two batch means (real labelled $1$, fake labelled $0$).
The generator's non-saturating loss is $\mathrm{BCE}(\ell_{\text{fake}},1)=-\text{mean}\log\sigma(\ell_{\text{fake}})=-\text{mean}\log D(G(z))$.
Using $\sigma'=\sigma(1-\sigma)$:
$$\frac{d}{d\ell}\log(1-\sigma(\ell))=\frac{-\sigma(1-\sigma)}{1-\sigma}=-\sigma(\ell),\qquad\frac{d}{d\ell}\big[-\log\sigma(\ell)\big]=-\frac{\sigma(1-\sigma)}{\sigma}=-(1-\sigma(\ell)).$$
When the discriminator confidently rejects fakes ($\ell_{\text{fake}}\ll0$), $\sigma(\ell)\approx0$: the minimax gradient $-\sigma(\ell)\approx0$ saturates, while the non-saturating gradient $-(1-\sigma(\ell))\approx-1$ stays strong.

**Rejected options.**

* **B** swaps the discriminator's labels (real $\to0$, fake $\to1$), which trains a discriminator that calls fakes real, the opposite of maximizing $V$.
* **C** gives the generator $\mathrm{BCE}(\ell_{\text{fake}},0)$, the discriminator's own fake-row term, so the generator would push $D(G(z))$ **down** and help the discriminator instead of fooling it; the game is not a shared objective.
* **D** is wrong twice: the players do not both minimize $V$ (the discriminator maximizes it), and the non-saturating loss is **not** $-V$, so $C(G)=-\log4+2\,\mathrm{JS}$ is not the generator's exact training objective.
* **E** has the gradient claim backwards: $\log(1-D(G(z)))$ has logit gradient $-\sigma(\ell)$, which is **smallest** (near $0$) when $D(G(z))\approx0$ — that is exactly the saturation problem.

**Gradients at $\ell=-3$.** $\sigma(-3)=1/(1+e^{3})=0.047426$.
Saturating (minimax) loss $\log(1-\sigma(\ell))$: gradient $-\sigma(-3)=-0.047426$.
Non-saturating loss $-\log\sigma(\ell)$: gradient $-(1-\sigma(-3))=-0.952574$, about $20$ times larger.

In [ ]:
import math

import torch
import torch.nn.functional as F

ell = -3.0
sig = 1.0 / (1.0 + math.exp(-ell))
grad_sat_formula = -sig
grad_ns_formula = -(1.0 - sig)

# autograd cross-check with numerically stable BCE-with-logits forms
l1 = torch.tensor(ell, dtype=torch.float64, requires_grad=True)
saturating = -F.binary_cross_entropy_with_logits(l1, torch.zeros_like(l1))   # log(1 - sigma(l))
(grad_sat,) = torch.autograd.grad(saturating, l1)
l2 = torch.tensor(ell, dtype=torch.float64, requires_grad=True)
non_saturating = F.binary_cross_entropy_with_logits(l2, torch.ones_like(l2))  # -log sigma(l)
(grad_ns,) = torch.autograd.grad(non_saturating, l2)
ANSWER = "A"
print(f"sigma(-3) = {sig:.6f}")
print(f"saturating grad {grad_sat.item():.6f}   non-saturating grad {grad_ns.item():.6f}")

### Answer check

In [ ]:
assert round(sig, 6) == 0.047426
assert round(grad_sat_formula, 6) == -0.047426
assert round(grad_ns_formula, 6) == -0.952574
assert math.isclose(grad_sat.item(), grad_sat_formula, abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(grad_ns.item(), grad_ns_formula, abs_tol=1e-12, rel_tol=1e-09)
assert abs(grad_ns.item()) > 20 * abs(grad_sat.item())
assert ANSWER == "A"